# Libaries and Preamble

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(1, './..')
import functions
from functions import read_squid_file
from functions import read_squid_file
import matplotlib.pyplot as plt
import os
from scipy.optimize import least_squares

In [ ]:
#=======================================================================
# matplotlib rc — academic / journal style
#=======================================================================
plt.rcParams.update({
    "font.family":       "serif",
    "font.size":         10,
    "axes.labelsize":    10,
    "axes.titlesize":    10,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "legend.fontsize":   9,
    "lines.linewidth":   1.0,
    "axes.linewidth":    0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.minor.width": 0.5,
    "ytick.minor.width": 0.5,
    "xtick.major.size":  4.0,
    "ytick.major.size":  4.0,
    "xtick.minor.size":  2.0,
    "ytick.minor.size":  2.0,
    "xtick.direction":   "in",
    "ytick.direction":   "in",
    "xtick.top":         True,
    "ytick.right":       True,
    "axes.grid":         True,
    "grid.color":        "#CCCCCC",
    "grid.linewidth":    0.4,
    "grid.linestyle":    "--",
    "grid.alpha":        0.6,
    "axes.axisbelow":    True,
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
    "savefig.dpi":       600,
    "savefig.bbox":      "tight",
    "savefig.facecolor": "white",
    "figure.dpi":        150,
    "legend.framealpha": 0.9,
    "legend.edgecolor":  "#AAAAAA",
    "legend.frameon":    True,
})

In [ ]:
# pb = read_squid_file(
#     r"C:\Users\olive\OneDrive - University of Bristol\Final Project\Data\Ollie_and_Ben\2026\10\ob002_ambient\FW_Various\2026-10-28_FeSe_ob002_MvsT_FW_Various_Field.dat"
# )
# pb.head()

In [ ]:
# plt.plot(pb["Temperature (K)"],pb["DC Moment Fixed Ctr (emu)"])
# plt.title("2026-10-28 FeSe (ob002) Moment vs Temperature FW at 5 Oe")
# plt.ylabel("DC Moment Fixed Ctr (emu)")
# plt.xlabel("Temperature (K)")

### Current Plan for Data Analysis 
- we need a method for looping throught a set number of files
- easiest way is to make a folder for the data 
- and then loop throught the folder and get all the raw data 


In [ ]:
# Source - https://stackoverflow.com/a/10378012
# Posted by anselm, modified by community. See post 'Timeline' for change history
# Retrieved 2026-10-15, License - CC BY-SA 4.0
# edited by ben thomposn

from pathlib import Path
runs = {}
pathlist = Path(
    r"../../data/2026/11"
).glob("**/*.dat")
for path in sorted(pathlist):
    run_number = path.stem[-5:]
    runs[run_number] = read_squid_file(str(path))
for run_number, data in runs.items():
    print(f"Run {run_number}")

In [ ]:
field_column = "Magnetic Field (Oe)"
field_labels = {}

for run_number, dataframe in runs.items():
    mean_field = dataframe[field_column].mean()
    field_labels[run_number] = f"{mean_field:.0f} Oe"

In [ ]:

for run_number, dataframe in runs.items():
    plt.plot(
        dataframe["Temperature (K)"],
        dataframe["DC Moment Fixed Ctr (emu)"],
        label=field_labels[run_number],
    )

plt.title("FeSe (sample: ob002) Moment vs Temperature FW at Various Fields")
plt.xlabel("Temperature (K)")
plt.ylabel("DC Moment Fixed Ctr (emu)")
plt.legend()
plt.show()

In [ ]:
# # Git Copilot 5.6 Luna usgeenrateed to  example code structure for fitting normal-state and transition regions of MvsT data. Code modified by Ollie Pentecost to suit Pb data.

# temperature_column = "Temperature (K)"
# moment_column = "DC Moment Fixed Ctr (emu)"
# field_column = "Magnetic Field (Oe)"

# fit_results = {}
# fig, ax = plt.subplots(figsize=(8, 5))
# colors = plt.get_cmap("viridis", len(runs))

# tc_list = []
# mean_field_list = []

# for plot_index, (run_number, dataframe) in enumerate(runs.items()):
#     data = dataframe[[temperature_column, moment_column]].dropna()
#     data = data.sort_values(temperature_column)
#     data = data.drop_duplicates(subset=temperature_column)

#     temperature = data[temperature_column].to_numpy()
#     moment = data[moment_column].to_numpy()
#     colour = colors(plot_index)

#     mean_field = dataframe[field_column].mean()
#     field_label = f"{mean_field:.0f} Oe"

#     # Normalise moment between the low- and high-temperature limits
#     m_low, m_high = np.percentile(moment, [5, 95])
#     moment_norm = (moment - m_low) / (m_high - m_low)

#     normal_mask = (
#         (moment_norm >= 0.75)
#         & (temperature >= np.percentile(temperature, 70))
#     )
#     transition_mask = (
#         (moment_norm >= 0.55)
#         & (moment_norm <= 0.75)
#     )

#     normal_m, normal_c = np.polyfit(
#         temperature[normal_mask],
#         moment[normal_mask],
#         1,
#     )
#     transition_m, transition_c = np.polyfit(
#         temperature[transition_mask],
#         moment[transition_mask],
#         1,
#     )

#     tc = (transition_c - normal_c) / (normal_m - transition_m)
#     tc_list.append(tc)
#     mc = normal_m * tc + normal_c
#     mean_field_list.append(mean_field)
#     fit_results[run_number] = {
#         "Mean field (Oe)": mean_field,
#         "Tc (K)": tc,
#         "Moment at Tc (emu)": mc,
#         "normal_slope": normal_m,
#         "normal_intercept": normal_c,
#         "transition_slope": transition_m,
#         "transition_intercept": transition_c,
#     }

#     ax.plot(
#         temperature,
#         moment,
#         color=colour,
#         linewidth=1.1,
#         label=field_label,
#         zorder=2,
#     )

#     normal_x = np.linspace(
#         temperature[normal_mask].min(),
#         temperature[normal_mask].max(),
#         100,
#     )
#     transition_x = np.linspace(
#         temperature[transition_mask].min(),
#         temperature[transition_mask].max(),
#         100,
#     )

#     ax.plot(
#         normal_x,
#         normal_m * normal_x + normal_c,
#         color="black",
#         linestyle=(0, (6, 3)),
#         linewidth=1.8,
#         label="Normal-state fit" if plot_index == 0 else "_nolegend_",
#         zorder=5,
#     )
#     ax.plot(
#         transition_x,
#         transition_m * transition_x + transition_c,
#         color="red",
#         linestyle=(0, (6, 3)),
#         linewidth=1.8,
#         label="Transition fit" if plot_index == 0 else "_nolegend_",
#         zorder=5,
#     )
#     ax.plot(
#         tc,
#         mc,
#         "o",
#         color=colour,
#         markersize=5,
#         markeredgecolor="black",
#         label=fr"$T_c={tc:.3f}$ K",
#         zorder=6,
#     )

#     print(f"Run {run_number}: mean field = {mean_field:.2f} Oe, Tc = {tc:.4f} K")

# ax.set_title("FeSe (sample: ob002) Moment vs Temperature FW at Various Fields with Normal-State and Transition Fits")
# ax.set_xlabel("Temperature (K)")
# ax.set_ylabel(moment_column)
# ax.legend(ncol=2)
# #plt.ylim(-0.00001, 0.00001)
# #plt.xlim(7.5, 9)
# plt.show()
# t
# pd.DataFrame(fit_results).T

In [ ]:
# Huber fitting technique devloped from above code alongside Copilot 5.6 Luna to suggest implmentation
# with some code AI generated
# Code modified by Ollie Pentecost to improve readability and styling data.


temperature_column = "Temperature (K)"
moment_column = "DC Moment Fixed Ctr (emu)"
field_column = "Magnetic Field (Oe)"


fit_results = {}

fig, ax = plt.subplots(figsize=(8, 5))

colors = plt.get_cmap("viridis",len(runs))


tc_list = []
mean_field_list = []


for plot_index, (run_number, dataframe) in enumerate(runs.items()):

    data = dataframe.copy()

    # Sort by temperature and remove duplicate temperatures
    data = data.sort_values(temperature_column)

    data = data.drop_duplicates(subset=temperature_column)

    # Extract temperature and moment arrays
    temperature = data[temperature_column].to_numpy(dtype=float)

    moment = data[moment_column].to_numpy(dtype=float)

    colour = colors(plot_index)

    # Calculate the mean applied field
    mean_field = dataframe[field_column].mean()

    # Normalise the moment between its lower and upper limits

    m_low, m_high = np.percentile(moment,[5, 95])

    moment_range = m_high - m_low

    moment_norm = ((moment - m_low) / moment_range)

    # Define normal-state and transition masks

    normal_temperature_limit = np.percentile(temperature,70,)


    normal_mask = ((moment_norm >= 0.95) & (temperature >= normal_temperature_limit))
    

    transition_mask = ((moment_norm >= 0.55) & (moment_norm < 0.95)
        & (temperature < normal_temperature_limit))

    normal_temperature = temperature[normal_mask]
    normal_moment = moment[normal_mask]

    transition_temperature = temperature[transition_mask]
    transition_moment = moment[transition_mask]

    # Initial ordinary least-squares normal-state fit
    # This is only used to initialise the Huber fit.
    
    initial_normal_m, initial_normal_c = np.polyfit(normal_temperature, normal_moment, 1)


    # Huber normal-state linear fit

    def normal_state_residuals(parameters):
        slope, intercept = parameters

        predicted_moment = (slope * normal_temperature + intercept)

        return predicted_moment - normal_moment

    # Calculate residuals from the initial linear fit
    initial_predicted_moment = (initial_normal_m * normal_temperature + initial_normal_c)

    initial_residuals = (normal_moment - initial_predicted_moment)

    # Estimate the noise using the median absolute deviation
    median_residual = np.median(initial_residuals)

    mad = np.median(np.abs(initial_residuals - median_residual))


    # Convert MAD to an approximate standard deviation
    f_scale = 1.4826 * mad

    # Prevent an invalid zero scale for very clean data
    if not np.isfinite(f_scale) or f_scale <= 0:
        f_scale = np.finfo(float).eps

    normal_fit = least_squares(
        normal_state_residuals,
        x0=[
            initial_normal_m,
            initial_normal_c,
        ],
        loss="huber",
        f_scale=f_scale,
    )


    normal_m, normal_c = normal_fit.x

    # Fit the transition region with a straight line
    
    transition_m, transition_c = np.polyfit(transition_temperature,transition_moment,1)

    # Find Tc from the intersection of the two fitted lines
    # Normal-state line:
    #     moment = normal_m * T + normal_c
    # Transition line:
    #     moment = transition_m * T + transition_c
    # At Tc, the two lines intersect.

    slope_difference = (normal_m - transition_m)

    tc = (transition_c - normal_c) / slope_difference

    field_label = f"{mean_field:.0f} Oe " rf"$T_c = {tc:.2f}$ K"

    # Calculate fitted lines for plotting

    temperature_fit = np.linspace(np.min(temperature),np.max(temperature),300)

    normal_background = (normal_m * temperature_fit + normal_c)

    transition_fit = (transition_m * temperature_fit + transition_c)

    # Calculate residuals from the normal-state background

    normal_background_all = (normal_m * temperature + normal_c)

    moment_residual = (moment - normal_background_all)

    # Store results

    fit_results[run_number] = {
        "mean_field_list": mean_field,
        "tc_list": tc,
        "normal_slope": normal_m,
        "normal_intercept": normal_c,
        "transition_slope": transition_m,
        "transition_intercept": transition_c,
        "normal_f_scale": f_scale,
        "n_normal": normal_temperature.size,
        "n_transition": transition_temperature.size,
    }

    if np.isfinite(tc):
        tc_list.append(tc)
        mean_field_list.append(mean_field)

    # Plot original data and fitted regions


    ax.scatter(temperature,moment,color=colour,s=1,alpha=0.75,label=field_label)

    ax.plot(temperature_fit,normal_background,color=colour,linestyle="--",linewidth=1.5,alpha=0.9)

    ax.plot(temperature_fit,transition_fit,color=colour,linestyle=":",linewidth=1.5,alpha=0.9)

    # Plot the fitted Tc point
    tc_moment = (normal_m * tc + normal_c)

    ax.scatter(tc,tc_moment,color=colour,edgecolor="black",s=45,zorder=5,)


ax.set_xlabel("Temperature (K)")

ax.set_ylabel("DC Moment Fixed Ctr (emu)")

ax.set_title("FeSe (sample: ob002) Moment vs Temperature FW at Various Fields with Normal-State and Transition Fits")

ax.grid(True,alpha=0.3)

ax.legend(title="Applied field",fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
tc_array = np.array(tc_list)
tcsqrd = tc_array ** 2
field_array = np.array(mean_field_list)

(m, c), cov = np.polyfit(tcsqrd, field_array, 1, cov=True)
m_err, c_err = np.sqrt(np.diag(cov))

Hc0 = c
Tc0_sqrd = -c / m
Tc0 = np.sqrt(Tc0_sqrd)

Hc0_err = c_err
Tc0__sq_err = Tc0_sqrd * np.sqrt((m_err / m) ** 2 + (c_err / c) ** 2)
Tc0_err = 0.5 * Tc0 * (Tc0__sq_err / Tc0_sqrd)

print(f"Tc0\u00b2 = {Tc0_sqrd:.2f} ± {Tc0__sq_err:.2f} K\u00b2")
print(f"Hc0 = {Hc0:.2f} ± {Hc0_err:.2f} Oe")
print(f"Tc0 = {Tc0:.2f} ± {Tc0_err:.2f} K")

#extrapolate the line
x_fit = np.linspace(0, tcsqrd.max() * 1.4, 100)

plt.figure(figsize=(6, 4))
plt.plot(tcsqrd, field_array, "o", label="Data")
plt.plot(x_fit, m * x_fit + c, "r--", label=f"Linear fit: H = {m:.2f} Tc² + {c:.2f}")
plt.scatter([0, Tc0_sqrd], [Hc0, 0], color="green", zorder=3, label="Intercepts")
plt.legend()
#plt.xlim(left=0)
plt.ylim(top=60, bottom=-30)
plt.title("2026-10-22 FeSe (ob002) FW at Various Fields: Hc vs Tc\u00b2 with extrapolated linear fit")
plt.ylabel("Mean Field (Oe)")
plt.xlabel("Critical Temperature Squared (Tc\u00b2)")

In [ ]:
import numpy as np
import bokeh 
from bokeh.layouts import column
from bokeh.io import push_notebook, show, output_notebook
from bokeh.layouts import row
from bokeh.layouts import column
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, HoverTool, TabPanel, Tabs, Span
from bokeh.palettes import Category20_20, Category10_10

output_notebook()
temperature_column = "Temperature (K)"
moment_column = "DC Moment Fixed Ctr (emu)"
field_column = "Magnetic Field (Oe)"

fit_results = {}
tc_list = []
mean_field_list = []

# Initialize Bokeh figure
p = figure(
    width=800,
    height=500,
    title="FeSe (sample: ob002) Moment vs Temperature FW at Various Fields with Normal-State and Transition Fits",
    x_axis_label="Temperature (K)",
    y_axis_label="DC Moment Fixed Ctr (emu)",
    x_range=(6, 9), 
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

# Generate a palette dynamically based on the number of runs
colors = viridis(len(runs))

for plot_index, (run_number, dataframe) in enumerate(runs.items()):
    data = dataframe.copy()
    data = data.sort_values(temperature_column).drop_duplicates(subset=temperature_column)

    temperature = data[temperature_column].to_numpy(dtype=float)
    moment = data[moment_column].to_numpy(dtype=float)
    colour = colors[plot_index]

    mean_field = dataframe[field_column].mean()

    m_low, m_high = np.percentile(moment, [5, 95])
    moment_range = m_high - m_low
    moment_norm = ((moment - m_low) / moment_range)

    normal_temperature_limit = np.percentile(temperature, 70)

    normal_mask = ((moment_norm >= 0.95) & (temperature >= normal_temperature_limit))
    transition_mask = ((moment_norm >= 0.55) & (moment_norm < 0.95) & (temperature < normal_temperature_limit))

    normal_temperature = temperature[normal_mask]
    normal_moment = moment[normal_mask]

    transition_temperature = temperature[transition_mask]
    transition_moment = moment[transition_mask]

    # Initial ordinary least-squares normal-state fit (initialization for Huber)
    initial_normal_m, initial_normal_c = np.polyfit(normal_temperature, normal_moment, 1)

    def normal_state_residuals(parameters):
        slope, intercept = parameters
        predicted_moment = (slope * normal_temperature + intercept)
        return predicted_moment - normal_moment

    initial_predicted_moment = (initial_normal_m * normal_temperature + initial_normal_c)
    initial_residuals = (normal_moment - initial_predicted_moment)

    median_residual = np.median(initial_residuals)
    mad = np.median(np.abs(initial_residuals - median_residual))
    f_scale = 1.4826 * mad

    if not np.isfinite(f_scale) or f_scale <= 0:
        f_scale = np.finfo(float).eps

    # Huber normal-state linear fit
    normal_fit = least_squares(
        normal_state_residuals,
        x0=[initial_normal_m, initial_normal_c],
        loss="huber",
        f_scale=f_scale,
    )
    normal_m, normal_c = normal_fit.x

    # Fit the transition region with a straight line
    transition_m, transition_c = np.polyfit(transition_temperature, transition_moment, 1)

    # Find Tc from intersection
    slope_difference = (normal_m - transition_m)
    tc = (transition_c - normal_c) / slope_difference
    
    # Bokeh does not support LaTeX natively in legend labels as easily as matplotlib
    field_label = f"{mean_field:.0f} Oe (Tc = {tc:.2f} K)"

    temperature_fit = np.linspace(np.min(temperature), np.max(temperature), 300)
    normal_background = (normal_m * temperature_fit + normal_c)
    transition_fit = (transition_m * temperature_fit + transition_c)

    fit_results[run_number] = {
        "mean_field_list": mean_field,
        "tc_list": tc,
        "normal_slope": normal_m,
        "normal_intercept": normal_c,
        "transition_slope": transition_m,
        "transition_intercept": transition_c,
        "normal_f_scale": f_scale,
        "n_normal": normal_temperature.size,
        "n_transition": transition_temperature.size,
    }

    if np.isfinite(tc):
        tc_list.append(tc)
        mean_field_list.append(mean_field)

    # Plot original data
    p.scatter(temperature, moment, color=colour, size=3, alpha=0.75, legend_label=field_label)

    # Plot normal background (dashed) and transition fit (dotted)
    p.line(temperature_fit, normal_background, color=colour, line_dash="dashed", line_width=1.5, alpha=0.9, legend_label=field_label)
    p.line(temperature_fit, transition_fit, color=colour, line_dash="dotted", line_width=1.5, alpha=0.9, legend_label=field_label)

    # Plot the fitted Tc point (drawn last to ensure it sits on top)
    tc_moment = (normal_m * tc + normal_c)
    p.scatter([tc], [tc_moment], color=colour, line_color="black", size=8, legend_label=field_label)


p.add_tools(HoverTool(tooltips=[("Temperature (K)", "@x{0.00}"),
                                ("DC Moment (emu)", "@y{0.000e}")]))

# Format legend behavior
p.add_layout(p.legend[0], 'right')
p.legend.click_policy = "hide"
p.legend.title = "Applied field"
p.legend.label_text_font_size = "8pt"

show(p)